# Part 1: Clustering on UCI Spam Dataset
## Assignment 4 — K-Center (Farthest-First Traversal) and K-Means++


In [1]:
import time
import numpy as np
from pyspark import SparkContext, SparkConf
from pyspark.mllib.linalg import Vectors

In [2]:
# Initialize Spark (use /DATA for temp to avoid full root partition)
import os
os.environ['SPARK_LOCAL_DIRS'] = '/DATA/suraj/m2/spark_tmp'

conf = SparkConf().setAppName("SpamClustering").setMaster("local[*]") \
    .set("spark.local.dir", "/DATA/suraj/m2/spark_tmp")
sc = SparkContext.getOrCreate(conf=conf)

26/04/18 13:36:27 WARN Utils: Your hostname, dip resolves to a loopback address: 127.0.1.1; using 10.6.0.83 instead (on interface enp98s0f1)
26/04/18 13:36:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/18 13:36:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/18 13:36:28 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


## 1. `readVectorsSeq(filename)`
Reads a CSV file where each line is a point->returns a list of Vector.

In [3]:
def readVectorsSeq(filename):
    """Read a text file of comma-separated floats into a list of Spark MLlib Vectors."""
    vectors = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                values = [float(x) for x in line.split(',')]
                vectors.append(Vectors.dense(values))
    return vectors

In [4]:
def sqdist(a, b):
    """Compute squared Euclidean distance between two Vectors."""
    diff = a.toArray() - b.toArray()
    return float(np.dot(diff, diff))

In [5]:
# Load dataset
DATA_PATH = "Q1- UCI Spam clustering/spambase.data"
P = readVectorsSeq(DATA_PATH)
print(f"Loaded {len(P)} points, each with {len(P[0])} dimensions")
print(f"First point (truncated): {P[0].toArray()[:5]}...")

Loaded 4601 points, each with 58 dimensions
First point (truncated): [0.   0.64 0.64 0.   0.32]...


## 2. `kcenter(P, k)` — Farthest-First Traversal


In [6]:
def kcenter(P, k):
    n = len(P)
    # Pick the first center arbitrarily (index 0)
    centers = [P[0]]

    # Parallelize points with their indices for distributed computation
    points_rdd = sc.parallelize([(i, P[i]) for i in range(n)]).cache()

    # min_dist[i] = squared distance from P[i] to its closest center so far
    min_dist = np.full(n, np.inf)

    for _ in range(1, k):
        # Broadcast the new center and current min distances to all workers
        new_center_bc = sc.broadcast(centers[-1])
        min_dist_bc = sc.broadcast(min_dist.copy())

        # Compute updated min distances in parallel using Spark RDD
        result = points_rdd.map(
            lambda item: (item[0], min(sqdist(item[1], new_center_bc.value), min_dist_bc.value[item[0]]))
        ).collect()

        # Update local min_dist array
        for idx, d in result:
            min_dist[idx] = d

        # Pick the point with the maximum distance to its closest center
        farthest_idx = int(np.argmax(min_dist))
        centers.append(P[farthest_idx])

        # Clean up broadcast variables
        new_center_bc.unpersist()
        min_dist_bc.unpersist()

    points_rdd.unpersist()
    return centers

## 3. `kmeansPP(P, k)` — K-Means++ Initialization

Time complexity:** $O(|P| \times k)$

In [7]:
def kmeansPP(P, k):
    
    n = len(P)
    rng = np.random.default_rng(seed=42)

    # Step 1: Choose the first center uniformly at random
    first_idx = rng.integers(0, n)
    centers = [P[first_idx]]

    # Parallelize points with their indices for distributed computation
    points_rdd = sc.parallelize([(i, P[i]) for i in range(n)]).cache()

    # min_dist[i] = squared distance from P[i] to its closest center
    min_dist = np.full(n, np.inf)

    for _ in range(1, k):
        # Broadcast the new center and current min distances to all workers
        new_center_bc = sc.broadcast(centers[-1])
        min_dist_bc = sc.broadcast(min_dist.copy())

        # Compute updated min distances in parallel using Spark RDD
        result = points_rdd.map(
            lambda item: (item[0], min(sqdist(item[1], new_center_bc.value), min_dist_bc.value[item[0]]))
        ).collect()

        # Update local min_dist array
        for idx, d in result:
            min_dist[idx] = d

        # Choose next center with probability proportional to D(x)
        total = min_dist.sum()
        if total == 0:
            # All points are already centers; pick randomly
            idx = rng.integers(0, n)
        else:
            probs = min_dist / total
            idx = rng.choice(n, p=probs)

        centers.append(P[idx])

        # Clean up broadcast variables
        new_center_bc.unpersist()
        min_dist_bc.unpersist()

    points_rdd.unpersist()
    return centers

## 4. `kmeansObj(P, C)` — K-Means Objective



In [8]:
def kmeansObj(P, C):
    """
    Compute the average squared distance of points in P from their closest center in C.
    Uses Spark RDD map-reduce for parallel computation.
    This is the k-means objective divided by |P|.
    """
    # Parallelize points and broadcast centers to all workers
    points_rdd = sc.parallelize(P)
    centers_bc = sc.broadcast(C)

    # Map each point to its min squared distance, then sum via reduce
    total_dist = points_rdd.map(
        lambda p: min(sqdist(p, c) for c in centers_bc.value)
    ).reduce(lambda a, b: a + b)

    centers_bc.unpersist()
    return total_dist / len(P)

## Experiments



In [9]:
# Parameters
k = 10
k1 = 50  # k1 > k, used as coreset size

print(f"Parameters: k = {k}, k1 = {k1}")
print(f"Dataset: {len(P)} points, {len(P[0])} dimensions")
print("=" * 60)

Parameters: k = 10, k1 = 50
Dataset: 4601 points, 58 dimensions


### Experiment 1: kcenter(P, k) — Farthest-First Traversal

In [10]:
# Experiment 1: Run kcenter(P, k) and measure running time
start = time.time()
C_kcenter = kcenter(P, k)
elapsed_kcenter = time.time() - start

obj_kcenter = kmeansObj(P, C_kcenter)
print(f"kcenter(P, {k}) running time: {elapsed_kcenter:.4f} seconds")
print(f"kcenter(P, {k}) kmeansObj: {obj_kcenter:.4f}")

kcenter(P, 10) running time: 15.7279 seconds
kcenter(P, 10) kmeansObj: 95359.2241


### Experiment 2: kmeansPP(P, k) → kmeansObj(P, C)

In [11]:
# Experiment 2: Run kmeansPP(P, k) and compute kmeansObj
start = time.time()
C_kmeanspp = kmeansPP(P, k)
elapsed_kmeanspp = time.time() - start

obj_kmeanspp = kmeansObj(P, C_kmeanspp)
print(f"kmeansPP(P, {k}) running time: {elapsed_kmeanspp:.4f} seconds")
print(f"kmeansPP(P, {k}) kmeansObj: {obj_kmeanspp:.4f}")

kmeansPP(P, 10) running time: 6.5240 seconds
kmeansPP(P, 10) kmeansObj: 26528.1248


### Experiment 3: Coreset approach — kcenter(P, k1) → kmeansPP(X, k) → kmeansObj(P, C)


In [12]:
# Experiment 3: Coreset approach
# Step 1: kcenter(P, k1) to get k1 centers as coreset X
start = time.time()
X = kcenter(P, k1)
elapsed_coreset_kcenter = time.time() - start
print(f"kcenter(P, {k1}) running time: {elapsed_coreset_kcenter:.4f} seconds")
print(f"Coreset size: {len(X)}")

# Step 2: kmeansPP(X, k) to extract k centers from the coreset
start = time.time()
C_coreset = kmeansPP(X, k)
elapsed_coreset_kmeanspp = time.time() - start
print(f"kmeansPP(X, {k}) running time: {elapsed_coreset_kmeanspp:.4f} seconds")

# Step 3: Evaluate kmeansObj(P, C)
obj_coreset = kmeansObj(P, C_coreset)
print(f"kmeansObj(P, C_coreset) = {obj_coreset:.4f}")

kcenter(P, 50) running time: 29.6421 seconds
Coreset size: 50


kmeansPP(X, 10) running time: 5.1978 seconds
kmeansObj(P, C_coreset) = 350760.4342


### Summary Comparison

In [13]:
# Summary table
print("=" * 60)
print(f"{'Method':<35} {'kmeansObj':>12} {'Time (s)':>10}")
print("-" * 60)
print(f"{'kcenter(P, k=' + str(k) + ')':<35} {obj_kcenter:>12.4f} {elapsed_kcenter:>10.4f}")
print(f"{'kmeansPP(P, k=' + str(k) + ')':<35} {obj_kmeanspp:>12.4f} {elapsed_kmeanspp:>10.4f}")
print(f"{'Coreset: kcenter→kmeansPP':<35} {obj_coreset:>12.4f} {elapsed_coreset_kcenter + elapsed_coreset_kmeanspp:>10.4f}")
print("=" * 60)
print()
print("Observations:")
print(f"  - kmeansPP objective ({obj_kmeanspp:.4f}) is typically lower than kcenter ({obj_kcenter:.4f})")
print(f"    because kmeans++ optimizes for average squared distance.")
print(f"  - The coreset approach (objective={obj_coreset:.4f}) runs kmeans++ on only {k1} points")
print(f"    instead of {len(P)}, making it much faster while still achieving competitive quality.")

Method                                 kmeansObj   Time (s)
------------------------------------------------------------
kcenter(P, k=10)                      95359.2241    15.7279
kmeansPP(P, k=10)                     26528.1248     6.5240
Coreset: kcenter→kmeansPP            350760.4342    34.8399

Observations:
  - kmeansPP objective (26528.1248) is typically lower than kcenter (95359.2241)
    because kmeans++ optimizes for average squared distance.
  - The coreset approach (objective=350760.4342) runs kmeans++ on only 50 points
    instead of 4601, making it much faster while still achieving competitive quality.


### Experiment 4: Effect of varying k1 on coreset quality

We test how increasing `k1` (coreset size from kcenter) improves the final kmeansObj when running kmeans++ on the coreset.

In [14]:
# Varying k1 to see the effect on coreset quality
k1_values = [20, 50, 100, 200, 500]
print(f"Fixed k = {k}, varying k1:")
print(f"{'k1':>5}  {'kmeansObj':>12}  {'Total Time (s)':>15}")
print("-" * 40)

for k1_val in k1_values:
    start = time.time()
    X_var = kcenter(P, k1_val)
    C_var = kmeansPP(X_var, k)
    elapsed = time.time() - start
    obj_var = kmeansObj(P, C_var)
    print(f"{k1_val:>5}  {obj_var:>12.4f}  {elapsed:>15.4f}")

print()
print(f"Direct kmeansPP(P, k={k}) objective for reference: {obj_kmeanspp:.4f}")

Fixed k = 10, varying k1:
   k1     kmeansObj   Total Time (s)
----------------------------------------


   20  1027453.5832          16.2200


   50   350760.4342          33.0209


  100   742232.4122          61.8513


  200   253750.0176         117.9689


  500   138558.1175         288.3571

Direct kmeansPP(P, k=10) objective for reference: 26528.1248


In [15]:
# Clean up Spark context
sc.stop()